# FuzeAI on a Kaggle GPU (personal endpoint)

This notebook builds GPU-enabled `llama.cpp`, serves the published FuzeAI GGUF, and opens a temporary HTTPS tunnel. Use a Kaggle GPU accelerator and enable Internet in Notebook settings. Run the cells from top to bottom.

**Security:** this no-key endpoint is public to anyone who obtains the temporary URL. Do not post or share it. Stop the notebook when finished. Kaggle and the tunnel provider can still impose infrastructure/session limits.

In [ ]:
import subprocess, os, pathlib, time, urllib.request
print(subprocess.run(['nvidia-smi'], text=True, capture_output=True).stdout[:4000])
assert pathlib.Path('/kaggle').exists(), 'Run this notebook on Kaggle.'


In [ ]:
# Build llama.cpp with CUDA support for the Kaggle GPU.
subprocess.run('git clone --depth 1 https://github.com/ggml-org/llama.cpp /kaggle/working/llama.cpp', shell=True, check=True)
subprocess.run('cmake -S /kaggle/working/llama.cpp -B /kaggle/working/llama.cpp/build -DGGML_CUDA=ON -DCMAKE_BUILD_TYPE=Release', shell=True, check=True)
subprocess.run('cmake --build /kaggle/working/llama.cpp/build --config Release -j 4 --target llama-server', shell=True, check=True)
print('llama.cpp GPU server built.')


In [ ]:
# Start the model server. llama.cpp downloads the GGUF from Hugging Face on first launch.
server_log = open('/kaggle/working/fuzeai-server.log', 'w')
server = subprocess.Popen([
    '/kaggle/working/llama.cpp/build/bin/llama-server',
    '-hf', 'r7194714R/fuzeai-qwen-7b-gguf:Q4_K_M',
    '--host', '127.0.0.1', '--port', '8080', '-c', '4096', '-ngl', '99'
], stdout=server_log, stderr=subprocess.STDOUT)
ready = False
for _ in range(180):
    if server.poll() is not None:
        print(pathlib.Path('/kaggle/working/fuzeai-server.log').read_text(errors='replace')[-5000:])
        raise RuntimeError('llama-server exited before becoming ready; inspect the log above.')
    try:
        with urllib.request.urlopen('http://127.0.0.1:8080/health', timeout=2) as response:
            if response.status == 200:
                ready = True
                break
    except Exception:
        time.sleep(2)
if not ready:
    print(pathlib.Path('/kaggle/working/fuzeai-server.log').read_text(errors='replace')[-5000:])
    raise TimeoutError('Model server did not become ready within 6 minutes.')
print('FuzeAI model server is ready at http://127.0.0.1:8080')


In [ ]:
# Install Cloudflare's temporary HTTPS tunnel client.
subprocess.run('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -O /kaggle/working/cloudflared.deb', shell=True, check=True)
subprocess.run('dpkg -i /kaggle/working/cloudflared.deb', shell=True, check=True)
print('cloudflared installed.')


In [ ]:
# Keep this cell running. Copy the https://....trycloudflare.com URL from its output.
# The URL is temporary and anyone who gets it can send requests to your model.
tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8080'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)
try:
    for line in tunnel.stdout:
        print(line, end='')
except KeyboardInterrupt:
    tunnel.terminate()
    raise


## Connect FuzeCLI on your Windows computer

Copy the HTTPS `trycloudflare.com` address printed above. In PowerShell, run:

```powershell
aicli fuzeai connect https://YOUR-URL.trycloudflare.com
aicli fuzeai status
aicli chat
```

Run `aicli fuzeai disconnect` to reset FuzeCLI to the local endpoint. The Kaggle notebook must remain running while you use it. When done, interrupt the tunnel cell and stop the Kaggle session. This temporary tunnel is for personal testing, not permanent hosting.